# Label Generator — Triple Barrier Method
Applies ternary `Target` labels to every ticker in `data/merged_raw/` and writes
the result to `data/labeled/`.

| Barrier | Condition | Label |
|---|---|---|
| Take-Profit (TP) | Forward return ≥ `tp_pct` hits first | `1` |
| Stop-Loss (SL) | Forward return ≤ `sl_pct` hits first | `-1` |
| Time expiry | Neither barrier hit within `horizon` days | `0` |

The last `horizon` rows per ticker are dropped — they have no fully observable future window.

> **Implementation note:** The barrier check is fully vectorised via NumPy
> `sliding_window_view`. No `iterrows`, `apply`, or Python DataFrame loops are used.

## Cell 1 — Imports & Path Configuration

In [7]:
import os
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# ---------------------------------------------------------------------------
# Path constants
# ---------------------------------------------------------------------------
MERGED_DIR: str = "data/merged_raw"
LABELED_DIR: str = "data/labeled"

# ---------------------------------------------------------------------------
# Triple Barrier hyperparameters
# ---------------------------------------------------------------------------
HORIZON: int = 5       # trading days to look forward
TP_PCT: float = 0.03   # +3 % take-profit barrier
SL_PCT: float = -0.03  # -3 % stop-loss barrier

Path(LABELED_DIR).mkdir(parents=True, exist_ok=True)

print(f"Input  : {MERGED_DIR}")
print(f"Output : {LABELED_DIR}")
print(f"Params : horizon={HORIZON}  TP={TP_PCT:+.1%}  SL={SL_PCT:+.1%}")

Input  : data/merged_raw
Output : data/labeled
Params : horizon=5  TP=+3.0%  SL=-3.0%


## Cell 2 — Triple Barrier Labeling Function

**Vectorised implementation** — no `iterrows`, no Python DataFrame loops.

For each row `i`, the core steps are:
1. Build a `(N - horizon) × horizon` matrix of forward `adjusted_close` prices using `sliding_window_view`.
2. Compute percentage returns relative to each row's entry price in one broadcast operation.
3. Find the **first** TP hit and **first** SL hit per row with `np.argmax` on the boolean masks.
4. Label `1` where the TP index strictly precedes the SL index (and TP was actually reached).

In [8]:
def apply_triple_barrier(
    df: pd.DataFrame,
    horizon: int = HORIZON,
    tp_pct: float = TP_PCT,
    sl_pct: float = SL_PCT,
) -> pd.DataFrame:
    """
    Apply the Triple Barrier Method to a merged ticker DataFrame and return
    a copy with a ternary `Target` column appended.

    The last `horizon` rows are dropped because their full forward window
    extends beyond the available data.

    Parameters
    ----------
    df : pd.DataFrame
        Merged ticker DataFrame with a `Date` index (or column) and an
        `adjusted_close` column.
    horizon : int
        Number of trading days in the forward-looking window.
    tp_pct : float
        Take-profit barrier as a fractional return (e.g. 0.04 = +4 %).
    sl_pct : float
        Stop-loss barrier as a fractional return (e.g. -0.02 = -2 %).

    Returns
    -------
    pd.DataFrame
        Original DataFrame (minus the last `horizon` rows) with a new
        integer `Target` column:
          1  = TP hit first
         -1  = SL hit first (ties between TP and SL go to SL)
          0  = time expiry (neither barrier reached within horizon)

    Raises
    ------
    KeyError
        If `adjusted_close` is not present in `df`.
    ValueError
        If the DataFrame has fewer rows than `horizon + 1`.
    """
    if "adjusted_close" not in df.columns:
        raise KeyError(
            "'adjusted_close' column not found. "
            f"Available columns: {df.columns.tolist()}"
        )

    df = df.sort_values("Date").copy() if "Date" in df.columns else df.sort_index().copy()

    n: int = len(df)
    if n <= horizon:
        raise ValueError(
            f"DataFrame has only {n} rows — need at least horizon+1={horizon + 1}."
        )

    prices: np.ndarray = df["adjusted_close"].to_numpy(dtype=np.float64)
    n_labeled: int = n - horizon

    # ------------------------------------------------------------------
    # Build forward-price matrix: shape (n_labeled, horizon)
    #   Row i contains prices[i+1], prices[i+2], ..., prices[i+horizon]
    # sliding_window_view over prices[1:] gives windows of length `horizon`
    # starting at position i=0 → prices[1..horizon], i=1 → prices[2..horizon+1], etc.
    # ------------------------------------------------------------------
    future_prices: np.ndarray = sliding_window_view(prices[1:], horizon)[:n_labeled]
    # shape: (n_labeled, horizon)

    entry_prices: np.ndarray = prices[:n_labeled, np.newaxis]  # (n_labeled, 1) for broadcast

    returns: np.ndarray = (future_prices - entry_prices) / entry_prices
    # shape: (n_labeled, horizon)

    # ------------------------------------------------------------------
    # Locate first barrier hits along the horizon axis.
    # np.argmax on a bool array returns the index of the first True;
    # if no True exists it returns 0 — so we guard with .any().
    # We use `horizon` as a sentinel meaning "never hit".
    # ------------------------------------------------------------------
    NEVER: int = horizon  # sentinel

    tp_mask: np.ndarray = returns >= tp_pct   # (n_labeled, horizon)
    sl_mask: np.ndarray = returns <= sl_pct   # (n_labeled, horizon)

    tp_first: np.ndarray = np.where(tp_mask.any(axis=1), np.argmax(tp_mask, axis=1), NEVER)
    sl_first: np.ndarray = np.where(sl_mask.any(axis=1), np.argmax(sl_mask, axis=1), NEVER)

    # ------------------------------------------------------------------
    # Ternary label:
    #   1  — TP fires strictly before SL
    #  -1  — SL fires strictly before TP (ties → SL wins, conservative)
    #   0  — time expiry (neither barrier reached within horizon)
    # ------------------------------------------------------------------
    labels: np.ndarray = np.select(
        condlist=[
            (tp_first < sl_first) & (tp_first < NEVER),   # TP wins
            (sl_first <= tp_first) & (sl_first < NEVER),  # SL wins (ties → SL)
        ],
        choicelist=[1, -1],
        default=0,  # time expiry
    ).astype(np.int8)

    # Drop the last `horizon` rows (unlabelable) and attach the target
    labeled_df: pd.DataFrame = df.iloc[:n_labeled].copy()
    labeled_df["Target"] = labels
    return labeled_df

## Cell 3 — Process & Export All Tickers

In [9]:
input_files: list[str] = sorted(glob(os.path.join(MERGED_DIR, "*_merged.csv")))

if not input_files:
    print(f"[WARN] No merged CSVs found in {MERGED_DIR}. Run data_merger.ipynb first.")

skipped: list[str] = []
total: int = len(input_files)

print(f"Labeling {total} tickers  (horizon={HORIZON}, TP={TP_PCT:+.1%}, SL={SL_PCT:+.1%})\n")
print(f"{'Ticker':<8}  {'Rows':>6}  {'TP=1':>6}  {'SL=-1':>7}  {'Exp=0':>7}  {'TP Rate':>8}")
print("-" * 54)

for filepath in input_files:
    # Extract ticker symbol from filename: "AAPL_merged.csv" → "AAPL"
    ticker: str = os.path.basename(filepath).replace("_merged.csv", "")

    try:
        df: pd.DataFrame = pd.read_csv(filepath, parse_dates=["Date"])
    except (OSError, ValueError) as exc:
        print(f"[ERROR] {ticker}: could not load CSV — {exc}")
        skipped.append(ticker)
        continue

    try:
        labeled: pd.DataFrame = apply_triple_barrier(df, HORIZON, TP_PCT, SL_PCT)
    except (KeyError, ValueError) as exc:
        print(f"[ERROR] {ticker}: labeling failed — {exc}")
        skipped.append(ticker)
        continue

    out_path: str = os.path.join(LABELED_DIR, f"{ticker}_labeled.csv")
    labeled.to_csv(out_path, index=False)

    # ------------------------------------------------------------------
    # Label distribution logging
    # ------------------------------------------------------------------
    n_rows: int = len(labeled)
    n_tp: int = int((labeled["Target"] == 1).sum())
    n_sl: int = int((labeled["Target"] == -1).sum())
    n_exp: int = int((labeled["Target"] == 0).sum())
    tp_rate: float = n_tp / n_rows if n_rows > 0 else 0.0

    print(f"{ticker:<8}  {n_rows:>6}  {n_tp:>6}  {n_sl:>7}  {n_exp:>7}  {tp_rate:>7.1%}")

# Summary
print("-" * 54)
saved: int = total - len(skipped)
print(f"\nDone. {saved}/{total} tickers labeled and saved to '{LABELED_DIR}/'.")
if skipped:
    print(f"Skipped: {skipped}")

Labeling 49 tickers  (horizon=5, TP=+3.0%, SL=-3.0%)

Ticker      Rows    TP=1    SL=-1    Exp=0   TP Rate
------------------------------------------------------
AAPL         496     134      114      248    27.0%
ADBE         496     137      164      195    27.6%
AMAT         496     234      169       93    47.2%
AMC          488     166      261       61    34.0%
AMD          496     234      199       63    47.2%
AMZN         496     165      136      195    33.3%
ARM          495     248      208       39    50.1%
ASML         496     208      156      132    41.9%
AVGO         496     212      197       87    42.7%
COIN         496     225      245       26    45.4%
CRM          496     161      193      142    32.5%
CRWD         492     227      180       85    46.1%
DDOG         483     202      184       97    41.8%
GME          483     197      184      102    40.8%
GOOGL        496     174      129      193    35.1%
HOOD         411     214      169       28    52.1%
HUBS  